# Assignment 6: Capstone Modeling Lane  (FlyRank ML, Week 5)

**Status: BUILT (scaffold). Not yet DATA-VALIDATED. Not submitted.**

**Official brief received: NO**

The official assignment brief has NOT been provided to the builder yet. Do not treat this notebook as a completed submission. Paste the official brief below, then every requirement must be implemented and checked against it.

> Rules for this notebook: no client names, domains, URLs, private queries, credentials or raw exports. Clear all outputs before committing.

## Official brief (paste here)

_PLACEHOLDER: paste the full official brief text from the FlyRank dashboard, then list each requirement and deliverable as a checklist below._

- [ ] Requirement 1
- [ ] Requirement 2

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "flyrank_ml").exists())
sys.path.insert(0, str(ROOT / "src"))

from flyrank_ml.config import BlockedError, get_hf_token, load_config, load_env_file, require_configured
load_env_file()          # reads HF_TOKEN from your local .env (never paste the token here)
cfg = load_config()
print("Project root:", ROOT)
print("HF_TOKEN set:", bool(get_hf_token()))

In [ ]:
from flyrank_ml.data.loader import load_daily_from_warehouse

try:
    daily = load_daily_from_warehouse(cfg["data"])
    print("Loaded aggregated rows:", len(daily))
except BlockedError as exc:
    daily = None
    print("BLOCKED:", exc)

## Train model and compare with the baseline (same split)

In [ ]:
from flyrank_ml.pipeline import run_pipeline

if daily is not None:
    exp = {**cfg["experiment"], "algorithm": cfg["model"]["algorithm"], "model_params": cfg["model"]["model_params"]}
    exp.pop("lane", None); exp.pop("lane_status", None)
    results = run_pipeline(daily, ROOT / "reports", data_source="flyrank_real", cfg=exp)
    print({k: round(v, 3) for k, v in results["model_metrics"].items() if isinstance(v, float)})
    print({k: round(v, 3) for k, v in results["baseline_metrics"].items() if isinstance(v, float)})
else:
    print("Skipped: no real data loaded.")